# Generate Simulated Observation Data for KN with an Extinction
- Input data for SED classifier
- Author: Gregory S.H. Paek (24.01.13)

## Library

In [1]:
import glob
from astropy.table import Table, vstack
import numpy as np
from astropy import units as u
import pandas as pd
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')
import json

In [2]:
from astropy.time import Time

In [3]:
#	Plot presetting
import matplotlib.pyplot as plt
import matplotlib as mpl
#
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')

In [4]:
### Helper Functions
import sys
sys.path.append(os.path.join('..', 'src'))
from helper import makeSpecColors
from paths import *
from var import *
from sdtpy import *

In [5]:
import datetime
import os

# 현재 날짜와 시간을 가져옴
now = datetime.datetime.now()

# 원하는 형식으로 날짜와 시간을 문자열로 변환
# 예: '2023-01-30_15-30-00' 형식
formatted_date = now.strftime("%Y-%m-%d_%H-%M-%S")

In [6]:
# path_save = f"../data/Wollaeger+21/{formatted_date}"

In [7]:
# if not os.path.exists(path_save):
#     os.makedirs(path_save)
# print(path_save)

# Data

- Kilonova Model

In [8]:
import pickle

with open(os.path.join(RAW_WOLLAEGER_DATA, 'model.pkl'), 'rb') as f:
	interp = pickle.load(f)

In [9]:
mdarr = np.array([0.001, 0.003, 0.01, 0.03, 0.1])
vdarr = np.array([0.05, 0.15, 0.3])
mwarr = np.array([0.001, 0.003, 0.01, 0.03, 0.1])
vwarr = np.array([0.05, 0.15, 0.3])
lamarr = np.arange(3000., 12000.+5., 5.)
phasearr = np.array([0.125, 0.25, 0.5, 1.0, 2.0, 3.0])
# phasearr = np.array([0.125, 0.25, 0.5, 1.0, 2.0, 4.0])
# angarr = np.array([0, 30, 60, 90])
angarr = np.array([0, 15, 30, 45, 60, 75, 90])

In [10]:
import itertools

lamticks = np.arange(3000, 12000+1500, 1500)
z = 0.010
path_save = os.path.join(SPECTRA_WOLLAEGER_DATA, f"z{z:.3f}")
os.makedirs(path_save, exist_ok=True)

In [11]:
n_combinations = len(mdarr) * len(vdarr) * len(mwarr) * len(vwarr) * len(angarr) * len(phasearr)
print(n_combinations)

9450


In [14]:
metatbl = Table()
metatbl['number'] = np.arange(n_combinations)
metatbl['md'] = -99.
metatbl['vd'] = -99.
metatbl['mw'] = -99.
metatbl['vw'] = -99.
metatbl['ang'] = -99.
metatbl['phase'] = -99.
metatbl['spec'] = " "*300
metatbl['plot'] = " "*300

for nn, (md, vd, mw, vw, ang, phase) in enumerate(itertools.product(mdarr, vdarr, mwarr, vwarr, angarr, phasearr)):
    params = [md, vd, mw, vw, ang, phase]
    if nn % 25 == 0 or nn == n_combinations - 1:
        print(f"[{nn+1}/{n_combinations}] {params}")
    point = (*params, lamarr)
    tablename = os.path.join(path_save, f"kn_spectrum_md{md:.3f}_vd{vd:.3f}_mw{mw:.3f}_vw{vw:.3f}_ang{ang:.3f}_phase{phase:.3f}.fits")
    plotname = tablename.replace(".fits", ".png")

    #   Meta Table
    metatbl['md'][nn] = md
    metatbl['vd'][nn] = vd
    metatbl['mw'][nn] = mw
    metatbl['vw'][nn] = vw
    metatbl['ang'][nn] = ang
    metatbl['phase'][nn] = phase
    metatbl['spec'][nn] = tablename
    metatbl['plot'][nn] = plotname

    #   Table
    if (not os.path.exists(tablename)) & (not os.path.exists(plotname)):
        iflux = interp(point)
        (flam, lam) = apply_redshift_on_spectrum(spflam=iflux*flamunit, splam=lamarr*lamunit, z=z, z0=0, scale=True)

        outbl = Table()
        outbl['lam'] = lam.value
        outbl['flam'] = flam.value
        outbl.write(tablename)

        #   Figure
        fig = plt.figure(figsize=(8, 4))
        plt.plot(lam, flam, lw=2, alpha=0.8)
        plt.title(os.path.basename(tablename))
        plt.xlabel(r"Wavelength [$\rm \AA$]")
        plt.ylabel(r"$\rm f_{\lambda}$ [$\rm erg / \AA / s cm^2$]")
        plt.yscale("log")
        plt.xticks(lamticks)
        plt.savefig(plotname)
        plt.close(fig)
    #

    
meta_tablename = os.path.join(path_save, "meta.csv")
metatbl.write(meta_tablename, format='csv', overwrite=True)

[1/9450] [0.001, 0.05, 0.001, 0.05, 0, 0.125]
[26/9450] [0.001, 0.05, 0.001, 0.05, 60, 0.25]
[51/9450] [0.001, 0.05, 0.001, 0.15, 15, 0.5]
[76/9450] [0.001, 0.05, 0.001, 0.15, 75, 1.0]
[101/9450] [0.001, 0.05, 0.001, 0.3, 30, 2.0]
[126/9450] [0.001, 0.05, 0.001, 0.3, 90, 3.0]
[151/9450] [0.001, 0.05, 0.003, 0.05, 60, 0.125]
[176/9450] [0.001, 0.05, 0.003, 0.15, 15, 0.25]
[201/9450] [0.001, 0.05, 0.003, 0.15, 75, 0.5]
[226/9450] [0.001, 0.05, 0.003, 0.3, 30, 1.0]
[251/9450] [0.001, 0.05, 0.003, 0.3, 90, 2.0]
[276/9450] [0.001, 0.05, 0.01, 0.05, 45, 3.0]
[301/9450] [0.001, 0.05, 0.01, 0.15, 15, 0.125]
[326/9450] [0.001, 0.05, 0.01, 0.15, 75, 0.25]
[351/9450] [0.001, 0.05, 0.01, 0.3, 30, 0.5]
[376/9450] [0.001, 0.05, 0.01, 0.3, 90, 1.0]
[401/9450] [0.001, 0.05, 0.03, 0.05, 45, 2.0]
[426/9450] [0.001, 0.05, 0.03, 0.15, 0, 3.0]
[451/9450] [0.001, 0.05, 0.03, 0.15, 75, 0.125]
[476/9450] [0.001, 0.05, 0.03, 0.3, 30, 0.25]
[501/9450] [0.001, 0.05, 0.03, 0.3, 90, 0.5]
[526/9450] [0.001, 0.05, 0

: 